# 🔍 Tutorial — Endpoint Terpadu: Deteksi Jerawat + Rekomendasi Skincare

Notebook ini menunjukkan cara menggunakan endpoint `/acne/analyze` yang menggabungkan:
1. **Deteksi tingkat keparahan jerawat** dari foto wajah (CNN NNEW)
2. **Rekomendasi produk skincare** yang sesuai secara otomatis

Semua terjadi dalam **satu request** saja.

```
📸 Foto wajah
      ↓
POST /acne/analyze
      ↓
┌─────────────────────────────────────────┐
│  CNN NNEW Acne Level                    │
│  → Tingkat 0 / 1 / 2 / 3               │
│  → Confidence & probabilitas            │
├─────────────────────────────────────────┤
│  Skincare Recommender                   │
│  → Top-N produk yang cocok              │
│  → Bahan aktif, brand, peringatan       │
└─────────────────────────────────────────┘
```

| Tingkat | Kondisi | Fokus Skincare |
|---------|---------|----------------|
| 0 🟢 | Tidak ada/sangat ringan | Maintenance, hidrasi |
| 1 🔵 | Ringan | Jerawat ringan, pori, flek |
| 2 🟠 | Sedang | Jerawat sedang, bekas jerawat |
| 3 🔴 | Berat | Intensif + ⚠️ konsultasi dokter |

---
## ⚙️ Setup

In [1]:
import io, json, subprocess, time
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

BASE_URL     = 'http://127.0.0.1:8000'
LEVEL_COLORS = ['#639922', '#378ADD', '#BA7517', '#D85A30']
LEVEL_NAMES  = ['Tingkat 0', 'Tingkat 1', 'Tingkat 2', 'Tingkat 3']
LEVEL_EMOJI  = ['🟢', '🔵', '🟠', '🔴']

def pprint(resp):
    print(f'Status : {resp.status_code}')
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

print('✅ Setup selesai')

✅ Setup selesai


---
## 🖥️ Jalankan Server

Jalankan di terminal terpisah:
```bash
cd path/ke/project
uvicorn main:app --reload --port 8000
```

Atau dari notebook:

In [2]:
import subprocess, time, requests

server = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print('⏳ Menunggu server...', end='')
for _ in range(20):
    time.sleep(1)
    print('.', end='', flush=True)
    try:
        r = requests.get(f'{BASE_URL}/health', timeout=2)
        if r.status_code == 200:
            data = r.json()
            print('\n✅ Server siap!')
            print(f"   CNN model     : {'✅' if data.get('acne_cnn_model') else '❌'}")
            print(f"   Skincare model: {'✅' if data.get('acne_skin_bundle') else '❌'}")
            break
    except:
        pass
else:
    print('\n⚠️  Server belum merespons.')

⏳ Menunggu server.......................
⚠️  Server belum merespons.


---
## 1️⃣ Cek Info Endpoint Terpadu

In [ ]:
resp = requests.get(f'{BASE_URL}/acne/info')
data = resp.json()

print(f"Status CNN      : {'✅ Loaded' if data['cnn_loaded'] else '❌ Not loaded'}")
print(f"Status Skincare : {'✅ Loaded' if data['skincare_loaded'] else '❌ Not loaded'}")
print(f"Total produk    : {data.get('skincare_total_produk', '?')}")
print(f"Jenis kulit     : {data.get('skincare_kulit_options', [])}")
print(f"\nTingkat jerawat:")
for k, v in data.get('acne_levels', {}).items():
    print(f"  Tingkat {k}: {v}")

---
## 2️⃣ Demo: Analisis Satu Foto

In [ ]:
# ══════════════════════════════════════════════
#  Ganti ke path foto wajah Anda
IMAGE_PATH  = 'JPEGImages/levle2_86.jpg'  # contoh dari dataset NNEW
JENIS_KULIT = 'Berminyak'                  # Berminyak/Kering/Normal/Kombinasi/Sensitif
TOP_N       = 5
# ══════════════════════════════════════════════

import os
if os.path.exists(IMAGE_PATH):
    with open(IMAGE_PATH, 'rb') as f:
        resp = requests.post(
            f'{BASE_URL}/acne/analyze',
            files={'file': (os.path.basename(IMAGE_PATH), f, 'image/jpeg')},
            params={'jenis_kulit': JENIS_KULIT, 'top_n': TOP_N}
        )
else:
    # Gambar dummy untuk demo
    print('⚠️  File tidak ditemukan → menggunakan gambar dummy')
    dummy = Image.fromarray((np.random.rand(128,128,3)*255).astype(np.uint8))
    buf = io.BytesIO()
    dummy.save(buf, format='JPEG')
    buf.seek(0)
    resp = requests.post(
        f'{BASE_URL}/acne/analyze',
        files={'file': ('dummy.jpg', buf, 'image/jpeg')},
        params={'jenis_kulit': JENIS_KULIT, 'top_n': TOP_N}
    )

data = resp.json()
acne = data['acne']

print('═'*58)
print(f"  HASIL ANALISIS: {data['filename']}")
print('═'*58)
print(f"  Waktu inferensi  : {data['inference_time_ms']} ms")
print()
print(f"  🔬 DETEKSI JERAWAT")
print(f"  Tingkat          : {acne['acne_level']} — {acne['acne_label']}")
print(f"  Confidence       : {acne['confidence_pct']}")
print(f"  Deskripsi        : {acne['acne_deskripsi']}")
print(f"  Probabilitas     :")
for lvl, prob in acne['probabilities'].items():
    bar = '█' * int(prob * 30)
    print(f"    {lvl}: {bar} {prob:.3f}")
if acne['saran_dokter']:
    print(f"  ⚠️  DISARANKAN KONSULTASI DOKTER KULIT")
print()
print(f"  🧴 REKOMENDASI SKINCARE (kulit: {data['jenis_kulit']})")
print(f"  {'─'*52}")
print(f"  {'#':<4} {'Brand':<18} {'Produk':<28} {'Score'}")
print(f"  {'─'*52}")
for r in data['rekomendasi']:
    print(f"  {r['rank']:<4} {r['brand']:<18} {r['produk'][:28]:<28} {r['final_score']:.3f}")

In [ ]:
# ── Visualisasi hasil ─────────────────────────────────────────
data  = resp.json()
acne  = data['acne']
rekom = data['rekomendasi']
lvl   = acne['acne_level']

fig = plt.figure(figsize=(18, 6))
fig.suptitle(
    f"Hasil Analisis: {data['filename']}  |  Kulit: {data['jenis_kulit']}",
    fontsize=13, fontweight='bold'
)

# ── Panel 1: Tingkat jerawat (gauge) ──
ax1 = fig.add_subplot(1, 3, 1)
probs = list(acne['probabilities'].values())
alphas = [1.0 if i == lvl else 0.2 for i in range(4)]
bars = ax1.bar(LEVEL_NAMES, probs, color=LEVEL_COLORS, edgecolor='white', linewidth=1.5)
for bar, a, v in zip(bars, alphas, probs):
    bar.set_alpha(a)
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.01,
             f'{v:.2f}', ha='center', fontsize=9,
             fontweight='bold' if a == 1.0 else 'normal')
ax1.set_title(f'{LEVEL_EMOJI[lvl]} Tingkat {lvl} — {acne["confidence_pct"]}', fontsize=10)
ax1.set_ylabel('Probabilitas')
ax1.set_ylim([0, max(probs)*1.3])
ax1.grid(axis='y', alpha=0.3)

# ── Panel 2: Top rekomendasi (bar) ──
ax2 = fig.add_subplot(1, 3, 2)
labels = [f"{r['brand'][:8]}\n{r['produk'][:16]}" for r in rekom]
scores = [r['final_score'] for r in rekom]
bar_c  = plt.cm.Greens(np.linspace(0.4, 0.85, len(scores)))
b2 = ax2.barh(labels[::-1], scores[::-1], color=bar_c[::-1], edgecolor='white', height=0.6)
for bar, v in zip(b2, scores[::-1]):
    ax2.text(v+0.003, bar.get_y()+bar.get_height()/2,
             f'{v:.3f}', va='center', fontsize=9)
ax2.set_title('Top Rekomendasi Skincare', fontsize=10)
ax2.set_xlabel('Final Score')
ax2.set_xlim([0, max(scores)*1.3])
ax2.grid(axis='x', alpha=0.3)

# ── Panel 3: Detail produk top 1 ──
ax3 = fig.add_subplot(1, 3, 3)
ax3.axis('off')
top1 = rekom[0] if rekom else {}
detail_text = (
    f"🥇 Rekomendasi Terbaik\n\n"
    f"Brand    : {top1.get('brand','-')}\n"
    f"Produk   : {top1.get('produk','-')}\n"
    f"Jenis    : {top1.get('jenis_produk','-')}\n"
    f"Bahan    : {top1.get('bahan_aktif','-')}\n"
    f"Kulit    : {top1.get('untuk_kulit','-')}\n"
    f"Level    : {top1.get('level_utama','-')}\n"
    f"Skor     : {top1.get('final_score',0):.4f}\n\n"
    f"⚠️  {top1.get('peringatan','-')}"
)
ax3.text(0.05, 0.95, detail_text, transform=ax3.transAxes,
         va='top', fontsize=9.5, linespacing=1.8,
         bbox=dict(boxstyle='round', facecolor='#f5f5f0', alpha=0.8))
ax3.set_title('Detail Produk #1', fontsize=10)

plt.tight_layout()
plt.savefig('hasil_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: hasil_analisis.png')

---
## 3️⃣ Demo Batch: Banyak Foto Sekaligus

In [ ]:
# Simulasi 4 foto dari tingkat berbeda (dummy)
N_BATCH     = 4
JENIS_KULIT = 'Berminyak'

files = []
for i in range(N_BATCH):
    dummy = Image.fromarray((np.random.rand(128,128,3)*255).astype(np.uint8))
    buf   = io.BytesIO()
    dummy.save(buf, format='JPEG')
    buf.seek(0)
    files.append(('files', (f'foto_{i+1}.jpg', buf, 'image/jpeg')))

resp = requests.post(
    f'{BASE_URL}/acne/analyze/batch',
    files=files,
    params={'jenis_kulit': JENIS_KULIT, 'top_n': 3}
)
data = resp.json()

print(f"Total foto    : {data['total_images']}")
print(f"Total waktu   : {data['total_time_ms']} ms")
print(f"Rata-rata/foto: {data['total_time_ms']/data['total_images']:.1f} ms")
print()
print(f"{'Foto':<14} {'Tingkat':<12} {'Confidence':<14} {'Rekomendasi #1'}")
print('─'*65)
for r in data['results']:
    emoji = LEVEL_EMOJI[r['acne']['acne_level']]
    lvl   = r['acne']['acne_level']
    conf  = r['acne']['confidence_pct']
    top1  = r['rekomendasi'][0]['produk'][:20] if r['rekomendasi'] else '-'
    print(f"{r['filename']:<14} {emoji} Tingkat {lvl}    {conf:<14} {top1}")

In [ ]:
# ── Visualisasi batch ──────────────────────────────────────────
results = data['results']
n       = len(results)

fig, axes = plt.subplots(2, n, figsize=(5*n, 9))
fig.suptitle(f'Analisis Batch — {n} foto | Kulit: {JENIS_KULIT}',
             fontsize=13, fontweight='bold')

for col, r in enumerate(results):
    acne  = r['acne']
    lvl   = acne['acne_level']
    probs = list(acne['probabilities'].values())
    rekom = r['rekomendasi']

    # Baris 1: probabilitas tingkat
    alphas = [1.0 if i == lvl else 0.2 for i in range(4)]
    bars   = axes[0][col].bar(range(4), probs, color=LEVEL_COLORS, edgecolor='white')
    for bar, a in zip(bars, alphas):
        bar.set_alpha(a)
    axes[0][col].set_xticks(range(4))
    axes[0][col].set_xticklabels(['T0','T1','T2','T3'])
    axes[0][col].set_ylim([0, 1.15])
    axes[0][col].set_title(
        f"{r['filename']}\n{LEVEL_EMOJI[lvl]} Tingkat {lvl} ({acne['confidence_pct']})",
        fontsize=9
    )
    axes[0][col].grid(axis='y', alpha=0.3)

    # Baris 2: rekomendasi skincare
    if rekom:
        short  = [f"{x['brand'][:8]}\n{x['produk'][:14]}" for x in rekom]
        scores = [x['final_score'] for x in rekom]
        bc     = plt.cm.Greens(np.linspace(0.4, 0.8, len(scores)))
        axes[1][col].barh(short[::-1], scores[::-1], color=bc[::-1], edgecolor='white')
        axes[1][col].set_xlabel('Score', fontsize=8)
        axes[1][col].tick_params(axis='y', labelsize=7)
        axes[1][col].grid(axis='x', alpha=0.3)
    axes[1][col].set_title('Rekomendasi', fontsize=9)

    if acne['saran_dokter']:
        axes[1][col].text(0.5, -0.2, '⚠️ Saran: konsultasi dokter',
                          transform=axes[1][col].transAxes,
                          ha='center', fontsize=8, color='red', style='italic')

plt.tight_layout()
plt.savefig('hasil_batch.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Tersimpan: hasil_batch.png')

---
## 4️⃣ Pakai via curl (Terminal)

```bash
# Analisis satu foto
curl -X POST http://127.0.0.1:8000/acne/analyze \
     -F "file=@foto_wajah.jpg" \
     -G -d "jenis_kulit=Berminyak" -d "top_n=5"

# Analisis batch
curl -X POST http://127.0.0.1:8000/acne/analyze/batch \
     -F "files=@foto1.jpg" \
     -F "files=@foto2.jpg" \
     -G -d "jenis_kulit=Kering" -d "top_n=3"

# Info model
curl http://127.0.0.1:8000/acne/info
```

---
## 📌 Ringkasan Response `/acne/analyze`

```json
{
  "filename": "foto.jpg",
  "inference_time_ms": 120.5,
  "acne": {
    "acne_level": 2,
    "acne_label": "Tingkat 2 — Jerawat sedang",
    "acne_deskripsi": "Jerawat cukup banyak...",
    "confidence": 0.7812,
    "confidence_pct": "78.12%",
    "probabilities": {
      "Tingkat 0": 0.05,
      "Tingkat 1": 0.12,
      "Tingkat 2": 0.78,
      "Tingkat 3": 0.05
    },
    "saran_dokter": false
  },
  "jenis_kulit": "Berminyak",
  "total_rekomendasi": 5,
  "rekomendasi": [
    {
      "rank": 1,
      "brand": "Emina",
      "produk": "Niacinamide 10% + Zinc 1%",
      "jenis_produk": "Serum",
      "bahan_aktif": "Niacinamide",
      "untuk_kulit": "Sensitif",
      "level_utama": "Level 2",
      "peringatan": "Aman",
      "skor_dataset": 3,
      "final_score": 0.6241,
      "catatan": "Produk dapat dipertimbangkan untuk jerawat sedang..."
    }
  ]
}
```

In [ ]:
try:
    server.terminate()
    print('✅ Server dihentikan')
except NameError:
    print('Hentikan server dengan Ctrl+C di terminal.')